# 03. Density review

- 확인: 문서별 밀도, 두 검정, 환경 민감도, 원문 맥락
- 입력: 비공개 분석 산출물
- 공개: 원문 문장이 담긴 실행 출력 제거


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

research_root = os.environ.get("ESG_RESEARCH_ROOT")
if not research_root:
    raise RuntimeError("ESG_RESEARCH_ROOT에 비공개 ESG 연구자료 폴더를 지정하세요.")
comp_root = Path(research_root).expanduser() / "comp"
if not comp_root.is_dir():
    raise FileNotFoundError(comp_root)

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError(f"필요한 입력 파일이 없습니다: {path}")
    return path

output_dir = comp_root / "analysis" / "output"
sig_dir = output_dir / "significance"
tests = pd.read_csv(require_file(sig_dir / "significance_tests_all.csv"))
within = pd.read_csv(require_file(sig_dir / "significance_within_country.csv"))
e_sensitivity = pd.read_csv(require_file(sig_dir / "E_subcategory_density_sensitivity.csv"))
required = {"family", "comparison", "n_a", "n_b", "mean_density_a", "mean_density_b", "fdr_q_mwu", "fdr_q_fp"}
if not required.issubset(tests.columns):
    raise ValueError("국가 간 검정표의 열 구성이 예상과 다릅니다.")


## 국가 간 밀도 검정

- 양국 문서 수와 두 검정의 FDR 보정값 확인
- 두 검정의 유의성과 실질적 서술 차이 판정 구분


In [ ]:
view_cols = ["family", "comparison", "n_a", "n_b", "mean_density_a", "mean_density_b",
             "density_ratio", "rank_biserial", "fdr_q_mwu", "fdr_q_fp"]
comparison_focus = tests[tests["family"].isin(["카테고리별_밀도", "ESG하위범주_밀도", "target_field별_밀도", "page_type별_밀도"])].copy()
comparison_focus["두 검정 유의"] = (comparison_focus["fdr_q_mwu"] < 0.05) & (comparison_focus["fdr_q_fp"] < 0.05)
display(comparison_focus[view_cols + ["두 검정 유의"]])


## 환경 민감도·국가 내 비교

- 환경 오탐 가능 표현 제외 전후 비교
- 국가 내 검정과 국가 간 검정 구분


In [ ]:
sensitivity_cols = ["시나리오", "E언급수_kr", "E언급수_jp", "평균밀도_kr", "평균밀도_jp",
                    "평균밀도비_kr÷jp", "rank_biserial"]
display(e_sensitivity[[col for col in sensitivity_cols if col in e_sensitivity.columns]])
display(within[[col for col in ["family", "comparison", "n_a", "n_b", "fdr_q_kw", "fdr_q_mwu", "fdr_q_fp"]
                if col in within.columns]].head(30))


## 원문 맥락

- 환경·거버넌스 키워드를 지정해 문장 확인
- 기본값에서는 원문을 표시하지 않음


In [ ]:
review_lang = "kr"        # "kr" 또는 "jp"
review_category = "E"   # "E" 또는 "G"
review_alias = None      # 특정 alias를 입력해야 원문 문장을 표시합니다.

if review_lang not in ("kr", "jp") or review_category not in ("E", "G"):
    raise ValueError("언어는 kr/jp, 범주는 E/G 중에서 고르세요.")
mentions = pd.read_csv(require_file(output_dir / f"mentions_{review_lang}_raw.csv"), dtype={"doc_id": str})
selected = mentions[(mentions["is_selected_longest"] == True) & (mentions["category"] == review_category)]
if review_alias is None:
    display(selected.groupby(["alias", "canonical"], dropna=False).size()
            .sort_values(ascending=False).head(20).rename("언급 수").to_frame())
else:
    display(selected.loc[selected["alias"] == review_alias,
                         ["alias", "canonical", "sentence_text"]].head(12))


## 확인할 점

- S: 밀도 차이와 원문 근거
- E: 일반 용례와 민감도
- G: 양국 사전과 원문 맥락
- 학과·페이지 유형: 검정 결과와 문서 구성
